# Stochastic optimization theory — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Use noisy gradients whose average points the right way
Stochastic optimization trades exact gradients for cheap unbiased estimates. These five examples show unbiased minibatch gradients, variance shrinking with batch size, Robbins-Monro step decay, noisy convergence on a quadratic, and the bias introduced by sampling from the wrong distribution.

In [ ]:
# 1) minibatch gradient is unbiased
X=np.array([1.,2.,3.,4.]); y=2*X; w=0.; grads=2*X*(w*X-y); full=grads.mean(); batches=list(itertools.combinations(range(4),2)); means=[grads[list(b)].mean() for b in batches]
plt.figure(figsize=(5,3)); plt.bar(range(len(means)),means); plt.axhline(full,c='r'); plt.title('average minibatch gradient equals full gradient')
assert abs(np.mean(means)-full)<1e-12

In [ ]:
# 2) gradient variance shrinks with batch size
rng=np.random.default_rng(0); g=rng.normal(-2,3,1000); sizes=[1,5,20,100]; vars=[np.var([rng.choice(g,size=s,replace=False).mean() for _ in range(400)]) for s in sizes]
plt.figure(figsize=(5,3)); plt.loglog(sizes,vars,'-o'); plt.title('larger batches reduce variance')
assert vars[-1]<vars[0]

In [ ]:
# 3) Robbins-Monro steps decay but keep moving
T=np.arange(1,101); eta=1/T; plt.figure(figsize=(5,3)); plt.plot(T,eta); plt.title('eta_t=1/t: decreasing steps')
assert eta[-1]<eta[0] and eta.sum()>4

In [ ]:
# 4) SGD converges in expectation on a noisy quadratic
rng=np.random.default_rng(1); x=5.; path=[]
for t in range(1,201):
    path.append(x); grad=(x-1)+rng.normal(0,0.5); x=x-(0.8/t)*grad
plt.figure(figsize=(5,3)); plt.plot(path); plt.axhline(1,c='k',ls='--'); plt.title('noisy path settles near optimum')
assert abs(x-1)<0.3

In [ ]:
# 5) biased sampling gives the wrong gradient
true=np.array([-4.,-2.,0.,2.]); probs=np.array([.7,.1,.1,.1]); biased=(probs*true).sum(); uniform=true.mean()
plt.figure(figsize=(5,3)); plt.bar(['uniform gradient','biased sampler'],[uniform,biased]); plt.title('sampling distribution changes the target')
assert abs(uniform+1)<1e-12 and abs(biased+2.8)<1e-12